# Week 9 — Tree-Based Methods & Gradient Boosting
## Notebook 1: Decision Trees & CART

**Difficulty:** ⭐⭐⭐ (Intermediate) | **Estimated Time:** 3 hours

---

### Why This Matters

Decision trees are the foundation of some of the most powerful ML algorithms in use today — Random Forests, Gradient Boosted Trees (XGBoost, LightGBM, CatBoost). Understanding how a single tree works, where it shines, and where it breaks down is **essential** before building ensembles. They are also among the most **interpretable** models: a trained tree can be printed out and read by a non-technical stakeholder.

In this notebook you will:
- Understand the CART algorithm intuitively and mathematically
- Implement a decision tree **from scratch** in Python
- Verify your implementation against scikit-learn
- Visualize decision boundaries, depth effects, and leaf statistics
- Build both a regression tree (predicting house price) and a classification tree (above/below median)

---

### Real-World Analogy First

Picture a **real estate agent with a flowchart** pinned to their desk:

```
                [Is sqft > 2000?]
               /                 \
             YES                  NO
              |                    |
   [Good school district?]   [Age < 10 years?]
     /          \               /          \
   YES           NO           YES           NO
    |             |            |             |
 HIGH PRICE  MED PRICE    MED PRICE    LOW PRICE
```

Each question (node) splits houses into two groups. You follow the path that matches the house you're evaluating until you hit a final answer (leaf). The agent doesn't use a formula — they use a **series of yes/no questions**, each one narrowing down the possibilities.

That is exactly what a decision tree does. The CART algorithm automates the process of **choosing the best question at each node**.

---

### Plain-English Explanation

1. **Start with all your data** at the root node.
2. **Try every possible question**: for each feature and every possible threshold, split the data in two.
3. **Score each split**: which split makes the two resulting groups as *pure* as possible (low variance for regression, low impurity for classification)?
4. **Keep the best split**, create two child nodes, and repeat on each child.
5. **Stop** when a node is too small, you've hit max depth, or no split improves things.
6. **Leaf prediction**: for regression → mean of samples in leaf. For classification → majority class.

This is called **recursive binary splitting** and it is **greedy** — at each step it picks the locally best split without considering future splits.

---

### The CART Regression Formula

At each node, choose feature $j$ and threshold $t$ to minimise the **weighted Mean Squared Error**:

$$\text{Cost}(j, t) = \frac{n_\text{left}}{n} \cdot \text{MSE}_\text{left} + \frac{n_\text{right}}{n} \cdot \text{MSE}_\text{right}$$

where

$$\text{MSE}_\text{left} = \frac{1}{n_\text{left}} \sum_{i \in \text{left}} (y_i - \bar{y}_\text{left})^2$$

The algorithm searches over all $(j, t)$ pairs and picks the one with the lowest cost.

In [ ]:
# ── Imports & global settings ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Reproducibility and plot aesthetics
np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('All libraries loaded successfully.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

## Section 1 — Synthetic House Price Dataset

We generate **500 houses** with 6 features that realistically influence price:

| Feature | Description | Range |
|---|---|---|
| `sqft` | Interior square footage | 600 – 4 000 |
| `bedrooms` | Number of bedrooms | 1 – 6 |
| `bathrooms` | Number of bathrooms | 1 – 4 |
| `age` | Age of house in years | 0 – 60 |
| `school_score` | Local school rating (1–10) | 1 – 10 |
| `dist_center` | Distance to city center (km) | 1 – 30 |

Price is a **noisy linear function** of these features, so a tree won't overfit to pure noise.

In [ ]:
# ── 1. Generate synthetic house price dataset ──────────────────────────────────
np.random.seed(42)
n = 500

# Raw features (continuous + discrete)
sqft         = np.random.uniform(600, 4000, n)           # living area
bedrooms     = np.random.randint(1, 7, n).astype(float)  # 1-6 bedrooms
bathrooms    = np.random.randint(1, 5, n).astype(float)  # 1-4 bathrooms
age          = np.random.uniform(0, 60, n)               # house age in years
school_score = np.random.uniform(1, 10, n)               # school quality 1-10
dist_center  = np.random.uniform(1, 30, n)               # km to downtown

# Price model: dominant drivers + interaction + noise
# Base: sqft is king, school score and age also matter
price = (  80 * sqft
         + 15000 * bedrooms
         + 10000 * bathrooms
         - 1500  * age
         + 8000  * school_score
         - 3000  * dist_center
         + np.random.normal(0, 25000, n)   # realistic noise
        )
price = np.clip(price, 50000, None)  # no negative prices

# Assemble DataFrame
feature_names = ['sqft', 'bedrooms', 'bathrooms', 'age', 'school_score', 'dist_center']
df = pd.DataFrame({
    'sqft':         sqft,
    'bedrooms':     bedrooms,
    'bathrooms':    bathrooms,
    'age':          age,
    'school_score': school_score,
    'dist_center':  dist_center,
    'price':        price
})

print(f'Dataset shape: {df.shape}')
print(f'\nPrice statistics:')
print(df['price'].describe().apply(lambda x: f'${x:,.0f}'))

In [ ]:
# ── 1b. Quick EDA — feature correlations with price ───────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

colors = sns.color_palette('muted', 6)

for i, feat in enumerate(feature_names):
    axes[i].scatter(df[feat], df['price'] / 1000,
                    alpha=0.4, s=20, color=colors[i])
    # Correlation annotation
    corr = df[feat].corr(df['price'])
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Price ($000s)')
    axes[i].set_title(f'{feat}  (r = {corr:.2f})')

plt.suptitle('House Price vs Each Feature', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\nCorrelations with price (sorted):')
print(df.corr()['price'].drop('price').sort_values(ascending=False).to_string())

## Section 2 — CART Decision Tree From Scratch

### Tree Anatomy

```
Root node ──► [sqft ≤ 1850?]           ← entire dataset, first split
              /             \
    [age ≤ 25?]          [school ≤ 6?]  ← internal nodes
    /       \             /       \
 [leaf]   [leaf]      [leaf]    [leaf]  ← predictions live here
```

Each `DecisionTreeNode` stores:
- `feature_idx` — which column to test
- `threshold` — the cut-off value (≤ goes left)
- `left`, `right` — child nodes
- `value` — prediction (only used at leaf nodes)
- `is_leaf` — flag

The `_best_split` method is the heart of CART: it loops over every (feature, threshold) pair and keeps track of the one that maximally reduces the weighted impurity.

In [ ]:
# ── 2. CART Decision Tree — from-scratch implementation ───────────────────────

class DecisionTreeNode:
    """A single node in the decision tree."""
    def __init__(self):
        self.feature_idx  = None   # which feature to split on
        self.threshold    = None   # split value (≤ → left)
        self.left         = None   # left child node
        self.right        = None   # right child node
        self.value        = None   # leaf prediction
        self.is_leaf      = False  # True when this is a terminal node
        self.n_samples    = 0      # how many training samples reached here


class DecisionTreeScratch:
    """
    CART decision tree supporting regression (MSE) and
    binary/multi-class classification (Gini).
    """
    def __init__(self, max_depth=5, min_samples_split=2, task='regression'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.task = task          # 'regression' or 'classification'
        self.root = None
        self.n_leaves_ = 0        # populated after fit

    # ── Impurity measures ────────────────────────────────────────────────────
    def _impurity(self, y):
        """Weighted variance (regression) or Gini impurity (classification)."""
        if len(y) == 0:
            return 0.0
        if self.task == 'regression':
            # weighted variance = MSE × n (minimising this minimises MSE)
            return np.var(y) * len(y)
        else:
            _, counts = np.unique(y, return_counts=True)
            p = counts / len(y)
            return 1.0 - np.sum(p ** 2)   # Gini impurity

    # ── Find best (feature, threshold) split ─────────────────────────────────
    def _best_split(self, X, y):
        n, p = X.shape
        best_gain      = -np.inf
        best_feature   = None
        best_threshold = None
        parent_impurity = self._impurity(y)

        for j in range(p):                    # loop over features
            thresholds = np.unique(X[:, j])   # candidate split values
            for t in thresholds:
                left_mask  = X[:, j] <= t
                right_mask = ~left_mask
                # Skip splits that are too small
                if (left_mask.sum() < self.min_samples_split or
                        right_mask.sum() < self.min_samples_split):
                    continue
                # Gain = parent impurity – (left impurity + right impurity)
                gain = (parent_impurity
                        - self._impurity(y[left_mask])
                        - self._impurity(y[right_mask]))
                if gain > best_gain:
                    best_gain      = gain
                    best_feature   = j
                    best_threshold = t

        return best_feature, best_threshold, best_gain

    # ── Recursive tree construction ───────────────────────────────────────────
    def _build(self, X, y, depth):
        node = DecisionTreeNode()
        node.n_samples = len(y)

        # Stopping conditions → make a leaf
        stop = (depth >= self.max_depth
                or len(y) < self.min_samples_split
                or len(np.unique(y)) == 1)

        if not stop:
            feature, threshold, gain = self._best_split(X, y)
            stop = (feature is None or gain <= 0)

        if stop:
            node.is_leaf = True
            if self.task == 'regression':
                node.value = float(np.mean(y))
            else:
                node.value = int(np.bincount(y.astype(int)).argmax())
            self.n_leaves_ += 1
            return node

        # Internal node — recurse
        node.feature_idx = feature
        node.threshold   = threshold
        left_mask        = X[:, feature] <= threshold
        node.left  = self._build(X[left_mask],  y[left_mask],  depth + 1)
        node.right = self._build(X[~left_mask], y[~left_mask], depth + 1)
        return node

    def fit(self, X, y):
        """Train the tree on numpy arrays X (n×p) and y (n,)."""
        self.n_leaves_ = 0
        self.root = self._build(X, y, depth=0)
        return self

    # ── Prediction ────────────────────────────────────────────────────────────
    def _predict_one(self, x, node):
        """Traverse the tree for a single sample."""
        if node.is_leaf:
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X):
        """Predict for all rows in X."""
        return np.array([self._predict_one(x, self.root) for x in X])


print('DecisionTreeScratch class defined.')

In [ ]:
# ── 3. Train the scratch tree on house price regression ───────────────────────
X = df[feature_names].values
y = df['price'].values

# 80 / 20 train-test split (stratified is not needed for regression)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit scratch tree (max depth 4, regression)
tree_scratch = DecisionTreeScratch(max_depth=4, min_samples_split=5, task='regression')
tree_scratch.fit(X_train, y_train)

# Predict and compute RMSE
y_pred_scratch = tree_scratch.predict(X_test)
rmse_scratch = np.sqrt(mean_squared_error(y_test, y_pred_scratch))

print(f'Scratch tree  — depth=4 | leaves={tree_scratch.n_leaves_}')
print(f'Test RMSE     = ${rmse_scratch:,.0f}')
print(f'Relative RMSE = {rmse_scratch / y_test.mean() * 100:.1f}% of mean price')

In [ ]:
# ── 4. Verify scratch vs sklearn DecisionTreeRegressor ────────────────────────

# Train sklearn tree with identical hyperparameters
tree_sk = DecisionTreeRegressor(max_depth=4, min_samples_split=5, random_state=42)
tree_sk.fit(X_train, y_train)
y_pred_sk = tree_sk.predict(X_test)
rmse_sk = np.sqrt(mean_squared_error(y_test, y_pred_sk))

# Compare predictions numerically
max_diff = np.max(np.abs(y_pred_scratch - y_pred_sk))
mean_diff = np.mean(np.abs(y_pred_scratch - y_pred_sk))

print('=== Scratch vs sklearn comparison ===')
print(f'Scratch RMSE : ${rmse_scratch:,.0f}')
print(f'sklearn RMSE : ${rmse_sk:,.0f}')
print(f'Max absolute prediction difference : ${max_diff:,.0f}')
print(f'Mean absolute prediction difference: ${mean_diff:,.0f}')
print()
print('Note: small differences arise because sklearn uses optimised')
print('midpoint thresholds; our scratch version uses the raw unique values.')

# Scatter plot: scratch predictions vs sklearn predictions
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_pred_sk / 1000, y_pred_scratch / 1000,
           alpha=0.6, s=30, edgecolors='k', linewidths=0.3)
lims = [min(y_pred_sk.min(), y_pred_scratch.min()) / 1000,
        max(y_pred_sk.max(), y_pred_scratch.max()) / 1000]
ax.plot(lims, lims, 'r--', lw=2, label='Perfect agreement')
ax.set_xlabel('sklearn predictions ($000s)')
ax.set_ylabel('Scratch predictions ($000s)')
ax.set_title('Scratch Tree vs sklearn — Prediction Agreement')
ax.legend()
plt.tight_layout()
plt.show()

## Section 3 — Visualising the Fitted Tree

scikit-learn's `plot_tree` renders the entire tree structure. Each box shows:
- The splitting condition (e.g., `sqft ≤ 1728.5`)
- The MSE at that node
- The number of samples that reached it
- The predicted value (mean price)

Reading the tree from root to leaf traces exactly the logic of the real estate agent's flowchart.

In [ ]:
# ── 5. Visualise the fitted sklearn tree ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 9))

plot_tree(
    tree_sk,
    feature_names=feature_names,
    filled=True,           # colour nodes by predicted value
    rounded=True,          # nicer corners
    fontsize=8,
    max_depth=4,           # show full depth-4 tree
    ax=ax,
    precision=0            # no decimals in prices
)

ax.set_title(
    'CART Regression Tree — House Price Prediction (depth=4)\n'
    'Darker blue = higher predicted price',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

# Also print feature importances
print('\nFeature importances (sklearn):')
importances = pd.Series(tree_sk.feature_importances_, index=feature_names)
print(importances.sort_values(ascending=False).to_string())

In [ ]:
# ── 6. Decision boundary in 2D (sqft vs school_score) ─────────────────────────
# Fit a fresh tree using ONLY these 2 features for clean 2D visualization
feat_2d = ['sqft', 'school_score']
X2 = df[feat_2d].values
y2 = df['price'].values

tree_2d = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_2d.fit(X2, y2)

# Build a dense meshgrid to colour-code predicted price
x1_range = np.linspace(X2[:, 0].min(), X2[:, 0].max(), 250)
x2_range = np.linspace(X2[:, 1].min(), X2[:, 1].max(), 250)
xx1, xx2 = np.meshgrid(x1_range, x2_range)
grid_pred = tree_2d.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)

fig, ax = plt.subplots(figsize=(11, 7))
# Filled contour showing predicted price regions (axis-aligned rectangles)
cf = ax.contourf(xx1, xx2, grid_pred / 1000,
                 levels=20, cmap='RdYlGn', alpha=0.75)
plt.colorbar(cf, ax=ax, label='Predicted price ($000s)')

# Overlay training data points
sc = ax.scatter(X2[:, 0], X2[:, 1],
               c=y2 / 1000, cmap='RdYlGn',
               edgecolors='k', linewidths=0.4, s=20, alpha=0.8)

ax.set_xlabel('Square Footage (sqft)', fontsize=12)
ax.set_ylabel('School Score', fontsize=12)
ax.set_title(
    'Decision Boundary: CART Regression Tree (depth=4)\n'
    'Notice the axis-aligned rectangular regions — trees can only cut horizontally or vertically',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

## Section 4 — Bias–Variance Trade-off: Tree Depth

Depth is the primary hyperparameter controlling model complexity:

| Depth | Description | Bias | Variance |
|---|---|---|---|
| 1 (stump) | Single split | Very high | Very low |
| 3–5 | Balanced | Moderate | Moderate |
| ≥ 10 | Full tree | Very low | Very high |

When depth is unlimited, the tree **memorises** the training set (0 training error, poor test performance).
We look for the **elbow** in the validation curve — the depth where test RMSE stops improving.

In [ ]:
# ── 7. Tree depth vs training and validation RMSE ─────────────────────────────
depths = [1, 2, 3, 4, 5, 7, 10, 15, None]  # None = unlimited depth
depth_labels = [str(d) if d else 'None\n(full)' for d in depths]

train_rmse_list = []
val_rmse_list   = []
n_leaves_list   = []

for d in depths:
    t = DecisionTreeRegressor(max_depth=d, random_state=42)
    t.fit(X_train, y_train)

    tr_rmse  = np.sqrt(mean_squared_error(y_train, t.predict(X_train)))
    val_rmse = np.sqrt(mean_squared_error(y_test,  t.predict(X_test)))

    train_rmse_list.append(tr_rmse)
    val_rmse_list.append(val_rmse)
    n_leaves_list.append(t.get_n_leaves())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE curves
x_pos = range(len(depths))
axes[0].plot(x_pos, np.array(train_rmse_list)/1000, 'b-o', lw=2, label='Train RMSE')
axes[0].plot(x_pos, np.array(val_rmse_list)/1000,   'r-o', lw=2, label='Validation RMSE')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(depth_labels, fontsize=9)
axes[0].set_xlabel('Max Depth')
axes[0].set_ylabel('RMSE ($000s)')
axes[0].set_title('Depth vs RMSE — Bias–Variance Trade-off')
axes[0].legend()
axes[0].fill_between(x_pos,
                     np.array(train_rmse_list)/1000,
                     np.array(val_rmse_list)/1000,
                     alpha=0.12, color='purple', label='Gap (variance)')
# Mark optimal depth
best_idx = int(np.argmin(val_rmse_list))
axes[0].axvline(best_idx, color='green', ls='--', lw=1.5,
                label=f'Best depth = {depths[best_idx]}')
axes[0].legend(fontsize=9)

# Number of leaves
axes[1].bar(x_pos, n_leaves_list, color='steelblue', edgecolor='white')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(depth_labels, fontsize=9)
axes[1].set_xlabel('Max Depth')
axes[1].set_ylabel('Number of Leaf Nodes')
axes[1].set_title('Tree Size (Number of Leaves) vs Depth')

plt.tight_layout()
plt.show()

print('\nDepth | Train RMSE   | Val RMSE    | Leaves')
print('-' * 50)
for d, tr, va, nl in zip(depth_labels, train_rmse_list, val_rmse_list, n_leaves_list):
    print(f'{str(d):6s} | ${tr:>10,.0f} | ${va:>10,.0f} | {nl}')

## Section 5 — Classification Tree: Above/Below Median Price

We binarise the target:
- **Class 0** (below median): affordable houses
- **Class 1** (above median): expensive houses

The classification tree uses **Gini impurity** to find splits (details in Notebook 2).
At each leaf it predicts the **majority class** and stores class probabilities.

In [ ]:
# ── 8. Classification tree — above vs below median price ──────────────────────
median_price = np.median(y)
y_cls = (y > median_price).astype(int)    # 1 = above median, 0 = below

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Sklearn classification tree
clf_tree = DecisionTreeClassifier(max_depth=4, random_state=42, criterion='gini')
clf_tree.fit(X_tr_c, y_tr_c)
acc = accuracy_score(y_te_c, clf_tree.predict(X_te_c))

print(f'Median price    : ${median_price:,.0f}')
print(f'Class balance   : {y_cls.mean()*100:.1f}% above median')
print(f'Test accuracy   : {acc*100:.1f}%')

# Visualise classification tree
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    clf_tree,
    feature_names=feature_names,
    class_names=['Below Median', 'Above Median'],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
ax.set_title(
    f'CART Classification Tree — Above/Below Median Price (depth=4)\n'
    f'Test Accuracy = {acc*100:.1f}% | Orange = Above Median | Blue = Below Median',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8b. Classification decision boundary (sqft vs school_score) ───────────────
clf_2d = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_2d.fit(X2, y_cls)           # X2 = [sqft, school_score]

# Meshgrid predictions
Z_cls = clf_2d.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)

fig, ax = plt.subplots(figsize=(10, 6))
ax.contourf(xx1, xx2, Z_cls, levels=[-0.5, 0.5, 1.5],
            colors=['#AED6F1', '#F9E79F'], alpha=0.7)
colors_pts = ['#2471A3' if c == 0 else '#D4AC0D' for c in y_cls]
ax.scatter(X2[:, 0], X2[:, 1], c=colors_pts,
           edgecolors='k', linewidths=0.3, s=22, alpha=0.85)

# Legend
patch0 = mpatches.Patch(color='#2471A3', label='Below Median')
patch1 = mpatches.Patch(color='#D4AC0D', label='Above Median')
ax.legend(handles=[patch0, patch1], loc='upper left')
ax.set_xlabel('Square Footage')
ax.set_ylabel('School Score')
ax.set_title('Classification Decision Boundary — Above/Below Median Price\n'
             'Each rectangular region votes for one class',
             fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6 — Leaf Statistics: How Pure are the Leaves?

For a regression tree, a **pure** leaf means all samples have very similar prices — low variance.
Inspecting leaf statistics tells us:
1. How many samples each leaf serves (sample coverage)
2. How certain the prediction is (variance)
3. Which leaves are **over-fit** (1 sample each = pure by coincidence)

We walk the tree recursively and collect information at every leaf node.

In [ ]:
# ── 9. Leaf statistics — purity inspection ────────────────────────────────────
def collect_leaf_stats(node, X, y, feature_names):
    """Recursively walk the tree; return a list of dicts for each leaf."""
    stats = []

    def _walk(node, X_node, y_node, path):
        if node.is_leaf:
            stats.append({
                'path':       path,
                'n_samples':  len(y_node),
                'mean_price': np.mean(y_node),
                'std_price':  np.std(y_node),
                'min_price':  np.min(y_node),
                'max_price':  np.max(y_node),
            })
            return
        fname = feature_names[node.feature_idx]
        left_mask = X_node[:, node.feature_idx] <= node.threshold
        _walk(node.left,  X_node[left_mask],  y_node[left_mask],
              path + [f'{fname}≤{node.threshold:.0f}'])
        _walk(node.right, X_node[~left_mask], y_node[~left_mask],
              path + [f'{fname}>{node.threshold:.0f}'])

    _walk(node, X, y, [])
    return stats


# Use scratch tree trained on full training set
leaf_data = collect_leaf_stats(tree_scratch.root, X_train, y_train, feature_names)
leaf_df = pd.DataFrame(leaf_data).sort_values('mean_price')

print(f'Tree has {len(leaf_df)} leaves.  Training samples: {len(y_train)}')
print()
print('=== Leaf Statistics (sorted by mean price) ===')
for i, row in leaf_df.iterrows():
    path_str = ' → '.join(row['path'][-2:]) if row['path'] else 'root'
    print(f"  Leaf {i+1:2d}: n={row['n_samples']:3d}  "
          f"mean=${row['mean_price']/1000:6.1f}k  "
          f"std=${row['std_price']/1000:5.1f}k  "
          f"[{path_str}]")

In [ ]:
# ── 9b. Visualise leaf purity ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: mean price per leaf (sorted)
leaf_nums = range(1, len(leaf_df) + 1)
axes[0].barh(list(leaf_nums), leaf_df['mean_price'].values / 1000,
             xerr=leaf_df['std_price'].values / 1000,
             color='steelblue', alpha=0.8, capsize=3)
axes[0].set_xlabel('Predicted Price ($000s)')
axes[0].set_ylabel('Leaf index (sorted by mean)')
axes[0].set_title('Leaf Predictions with ±1 Std Dev')

# Right panel: scatter of std vs n_samples (reveals under-populated leaves)
scatter = axes[1].scatter(
    leaf_df['n_samples'],
    leaf_df['std_price'] / 1000,
    c=leaf_df['mean_price'] / 1000,
    cmap='plasma', s=90, edgecolors='k', linewidths=0.5
)
plt.colorbar(scatter, ax=axes[1], label='Mean price ($000s)')
axes[1].set_xlabel('Samples in Leaf')
axes[1].set_ylabel('Std Dev of Price in Leaf ($000s)')
axes[1].set_title('Leaf Purity: Std Dev vs Sample Count\n'
                  '(Lower-left = few samples, high purity — potential overfit)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 10. Scratch tree — classification mode ────────────────────────────────────
# Use the scratch tree in classification mode and compare with sklearn

tree_scratch_cls = DecisionTreeScratch(
    max_depth=4, min_samples_split=5, task='classification'
)
tree_scratch_cls.fit(X_tr_c, y_tr_c)
y_pred_cls_scratch = tree_scratch_cls.predict(X_te_c)

acc_scratch = accuracy_score(y_te_c, y_pred_cls_scratch)
acc_sklearn  = accuracy_score(y_te_c, clf_tree.predict(X_te_c))

print('=== Classification tree comparison ===')
print(f'Scratch accuracy : {acc_scratch*100:.1f}%')
print(f'sklearn  accuracy: {acc_sklearn*100:.1f}%')

# Confusion-matrix style breakdown
for label, preds in [('Scratch', y_pred_cls_scratch), ('sklearn', clf_tree.predict(X_te_c))]:
    tp = ((preds == 1) & (y_te_c == 1)).sum()
    tn = ((preds == 0) & (y_te_c == 0)).sum()
    fp = ((preds == 1) & (y_te_c == 0)).sum()
    fn = ((preds == 0) & (y_te_c == 1)).sum()
    print(f'\n{label}: TP={tp}, TN={tn}, FP={fp}, FN={fn}')

In [ ]:
# ── 11. Feature importances — regression and classification ───────────────────
# sklearn tracks impurity reduction (MDI = Mean Decrease in Impurity)
# across all splits where a feature was used, weighted by sample count.

reg_imp  = pd.Series(tree_sk.feature_importances_,  index=feature_names)
cls_imp  = pd.Series(clf_tree.feature_importances_, index=feature_names)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

reg_imp.sort_values().plot.barh(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Feature Importances — Regression Tree')
axes[0].set_xlabel('Mean Impurity Decrease (MDI)')

cls_imp.sort_values().plot.barh(ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Feature Importances — Classification Tree')
axes[1].set_xlabel('Mean Impurity Decrease (MDI)')

plt.suptitle('Which Features Does the Tree Rely On Most?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Regression importances (sorted):' )
print(reg_imp.sort_values(ascending=False).to_string())
print('\nClassification importances (sorted):')
print(cls_imp.sort_values(ascending=False).to_string())

In [ ]:
# ── 12. Residual analysis — what does the tree get wrong? ─────────────────────
y_pred_best = tree_sk.predict(X_test)
residuals   = y_test - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Actual vs predicted
axes[0].scatter(y_test / 1000, y_pred_best / 1000, alpha=0.5, s=25, color='steelblue')
lim = [y_test.min() / 1000, y_test.max() / 1000]
axes[0].plot(lim, lim, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($000s)')
axes[0].set_ylabel('Predicted Price ($000s)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Residuals vs predicted
axes[1].scatter(y_pred_best / 1000, residuals / 1000,
                alpha=0.5, s=25, color='darkorange')
axes[1].axhline(0, color='black', lw=1.5, ls='--')
axes[1].set_xlabel('Predicted Price ($000s)')
axes[1].set_ylabel('Residual ($000s)')
axes[1].set_title('Residuals vs Predicted\n'
                  '(Horizontal bands = discretised leaf predictions)')

# Residual distribution
axes[2].hist(residuals / 1000, bins=30, color='seagreen', edgecolor='white')
axes[2].axvline(0, color='black', lw=1.5, ls='--')
axes[2].set_xlabel('Residual ($000s)')
axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution')

plt.suptitle(f'CART Regression Tree — Residual Analysis  (Test RMSE = ${rmse_sk/1000:.0f}k)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Residual mean  : ${residuals.mean():,.0f} (should be near 0)')
print(f'Residual std   : ${residuals.std():,.0f}')
print(f'Max over-pred  : ${residuals.min():,.0f}')
print(f'Max under-pred : ${residuals.max():,.0f}')

In [ ]:
# ── 13. Overfitting demo — depth=100 memorises training set ───────────────────
# A very deep tree can assign a unique leaf to every training sample → 0 train error
tree_overfit = DecisionTreeRegressor(max_depth=None, random_state=42)  # unlimited
tree_overfit.fit(X_train, y_train)

train_rmse_overfit = np.sqrt(mean_squared_error(y_train, tree_overfit.predict(X_train)))
test_rmse_overfit  = np.sqrt(mean_squared_error(y_test,  tree_overfit.predict(X_test)))

print('=== Unlimited depth (max_depth=None) ===')
print(f'Number of leaves : {tree_overfit.get_n_leaves()}')
print(f'Max depth reached: {tree_overfit.get_depth()}')
print(f'Train RMSE       : ${train_rmse_overfit:,.0f}  ← memorised!')
print(f'Test  RMSE       : ${test_rmse_overfit:,.0f}  ← generalises poorly')
print()
print(f'Compare with depth=4:')
print(f'Train RMSE       : ${np.sqrt(mean_squared_error(y_train, tree_sk.predict(X_train))):,.0f}')
print(f'Test  RMSE       : ${rmse_sk:,.0f}')
print()
print('The overfit tree has many leaves with 1 sample each — it has memorised noise.')

In [ ]:
# ── 14. Effect of min_samples_leaf on generalisation ──────────────────────────
# min_samples_leaf controls the minimum samples allowed in any leaf.
# Higher value = more regularisation = simpler tree.

min_samples_vals = [1, 2, 5, 10, 20, 40, 80]
tr_rmse_ms = []
va_rmse_ms = []

for ms in min_samples_vals:
    t = DecisionTreeRegressor(max_depth=None, min_samples_leaf=ms, random_state=42)
    t.fit(X_train, y_train)
    tr_rmse_ms.append(np.sqrt(mean_squared_error(y_train, t.predict(X_train))))
    va_rmse_ms.append(np.sqrt(mean_squared_error(y_test,  t.predict(X_test))))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(min_samples_vals, np.array(tr_rmse_ms)/1000, 'b-o', lw=2, label='Train RMSE')
ax.plot(min_samples_vals, np.array(va_rmse_ms)/1000, 'r-o', lw=2, label='Validation RMSE')
ax.set_xlabel('min_samples_leaf')
ax.set_ylabel('RMSE ($000s)')
ax.set_title('min_samples_leaf as a Regularisation Parameter\n'
             '(Unlimited depth — regularisation comes only from min_samples_leaf)')
ax.legend()
plt.tight_layout()
plt.show()

best_ms_idx = np.argmin(va_rmse_ms)
print(f'Best min_samples_leaf = {min_samples_vals[best_ms_idx]}')
print(f'Best validation RMSE  = ${va_rmse_ms[best_ms_idx]:,.0f}')

## Summary: What You've Learned

| Concept | Key Takeaway |
|---|---|
| CART Algorithm | Recursively find the best (feature, threshold) split by minimising weighted MSE (regression) or Gini (classification) |
| Axis-aligned boundaries | Trees can only cut features horizontally/vertically — diagonal boundaries require many splits |
| Greedy search | Locally optimal splits may miss globally optimal solutions |
| Depth hyperparameter | Low depth = underfitting (high bias); high depth = overfitting (high variance) |
| Leaf prediction | Mean for regression; majority class for classification |
| Feature importance | Tracks total impurity reduction each feature provides across all splits |
| Regularisation | `max_depth`, `min_samples_leaf`, `min_samples_split` all constrain tree size |

**Next Up:** Notebook 2 dives into the mathematics of splitting criteria (Gini, Entropy, Variance Reduction) that power the CART algorithm's node-scoring step.

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** A decision tree with `max_depth=100` on 500 training samples achieves **0% training error**. What has it done, and why?

<details>
<summary>Show Answer</summary>

The tree has **memorised** the training set. With enough depth, CART can create a separate leaf for every individual training sample (or small group), each predicting exactly that sample's value. Because the predicted value equals the actual value at each leaf, training error is zero. This is a textbook case of **overfitting**: the model has learned the training data's noise along with its signal. On new, unseen data the performance will be poor because the memorised noise does not generalise. The fix is to limit `max_depth`, increase `min_samples_leaf`, or use post-pruning.

</details>

---

**Q2.** Decision trees learn "axis-aligned" decision boundaries. What type of boundary can they **NOT** easily learn?

<details>
<summary>Show Answer</summary>

Decision trees cannot easily learn **diagonal or curved decision boundaries**. Because every split is of the form `feature_j ≤ threshold`, the regions in feature space are always **axis-parallel rectangles**. A boundary like `sqft + school_score > 2000` (a diagonal line) would require many small rectangular approximations — deep tree, lots of splits, high variance. Contrast this with linear models (which can learn any hyperplane directly) or SVMs with RBF kernels (which can learn smooth curves). This is one reason ensembles of many shallow trees (Random Forests, gradient boosting) are more powerful — together they can approximate diagonal boundaries.

</details>

---

**Q3.** CART always picks the **single best split** at each step. This is a **greedy** algorithm. Why might greedy not be globally optimal?

<details>
<summary>Show Answer</summary>

Greedy algorithms make the locally best choice at each step without looking ahead. In a decision tree, the best split at the root might **not** lead to the best tree overall. Imagine a feature that creates a seemingly poor split at depth 0, but after that split, both children can be split extremely cleanly at depth 1. A greedy algorithm would skip it in favour of a marginally better split at depth 0 that leads to messier children. The globally optimal tree would require evaluating all $O(2^n)$ possible tree structures — computationally infeasible. This is why CART is a **heuristic**: it usually finds a good tree, but not necessarily the best one. Ensemble methods (bagging, boosting) partially compensate for this by combining many greedy trees.

</details>